In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, sqrt, pow

spark = SparkSession.builder.appName('BikeAnalysis').getOrCreate()

trips = spark.read.option('header', True).option('inferSchema', True).csv('trip.csv')
stations = spark.read.option('header', True).option('inferSchema', True).csv('station.csv')

trips.show(5)
stations.show(5)

+----+--------+---------------+--------------------+----------------+---------------+--------------------+--------------+-------+-----------------+--------+
|  id|duration|     start_date|  start_station_name|start_station_id|       end_date|    end_station_name|end_station_id|bike_id|subscription_type|zip_code|
+----+--------+---------------+--------------------+----------------+---------------+--------------------+--------------+-------+-----------------+--------+
|4576|      63|8/29/2013 14:13|South Van Ness at...|              66|8/29/2013 14:14|South Van Ness at...|            66|    520|       Subscriber|   94127|
|4607|      70|8/29/2013 14:42|  San Jose City Hall|              10|8/29/2013 14:43|  San Jose City Hall|            10|    661|       Subscriber|   95138|
|4130|      71|8/29/2013 10:16|Mountain View Cit...|              27|8/29/2013 10:17|Mountain View Cit...|            27|     48|       Subscriber|   97214|
|4251|      77|8/29/2013 11:29|  San Jose City Hall|      

# 1. Велосипед с максимальным временем пробега

In [ ]:
trips.orderBy(col('duration').desc()).select('bike_id', 'duration').show(1)

+-------+--------+
|bike_id|duration|
+-------+--------+
|    535|17270400|
+-------+--------+
only showing top 1 row


# 2. Наибольшее расстояние между станциями

In [ ]:
a = stations.alias('a')
b = stations.alias('b')

dist = a.crossJoin(b).withColumn(
    'distance',
    sqrt(pow(col('a.lat') - col('b.lat'), 2) + pow(col('a.long') - col('b.long'), 2))
)

dist.orderBy(col('distance').desc()).select(
    col('a.name').alias('station_1'),
    col('b.name').alias('station_2'),
    'distance'
).show(1, False)

+--------------------------+----------------------+------------------+
|station_1                 |station_2             |distance          |
+--------------------------+----------------------+------------------+
|SJSU - San Salvador at 9th|Embarcadero at Sansome|0.7058482821754397|
+--------------------------+----------------------+------------------+
only showing top 1 row


# 3. Путь велосипеда с максимальным временем пробега

In [ ]:
trips.orderBy(col('duration').desc()).select(
    'bike_id',
    'start_station_name',
    'end_station_name',
    'duration'
).show(1, False)

+-------+------------------------+----------------+--------+
|bike_id|start_station_name      |end_station_name|duration|
+-------+------------------------+----------------+--------+
|535    |South Van Ness at Market|2nd at Folsom   |17270400|
+-------+------------------------+----------------+--------+
only showing top 1 row


# 4. Количество велосипедов в системе

In [ ]:
print(trips.select('bike_id').distinct().count())

700


# 5. Пользователи, потратившие более 3 часов

In [ ]:
trips.groupBy('subscription_type').agg(
    sum('duration').alias('total_time')
).filter(col('total_time') > 10800).show()

+-----------------+----------+
|subscription_type|total_time|
+-----------------+----------+
|       Subscriber| 334407829|
|         Customer| 407873142|
+-----------------+----------+



# Экстра

In [ ]:
trips.groupBy('start_station_name').count().orderBy(col('count').desc()).show(5, False)

+---------------------------------------------+-----+
|start_station_name                           |count|
+---------------------------------------------+-----+
|San Francisco Caltrain (Townsend at 4th)     |49092|
|San Francisco Caltrain 2 (330 Townsend)      |33742|
|Harry Bridges Plaza (Ferry Building)         |32934|
|Embarcadero at Sansome                       |27713|
|Temporary Transbay Terminal (Howard at Beale)|26089|
+---------------------------------------------+-----+
only showing top 5 rows


In [ ]:
trips.groupBy("bike_id").agg(
    sum("duration").alias("total_time")
).filter("total_time > 10800").show()

+-------+----------+
|bike_id|total_time|
+-------+----------+
|    471|   1718831|
|    496|   1679568|
|    148|    332138|
|    463|   1722796|
|    540|   1752835|
|    392|   1789476|
|    623|   2037219|
|    243|    307458|
|    516|   1896751|
|     31|    407907|
|    580|   1034382|
|    137|   1529200|
|    251|   1282980|
|    451|   1695574|
|     85|   1214769|
|    458|   1647080|
|     65|    216922|
|    588|    266415|
|    255|    396395|
|     53|    226389|
+-------+----------+
only showing top 20 rows


In [ ]:
trips.groupBy("start_station_name").count().orderBy("count", ascending=False).show(5)

+--------------------+-----+
|  start_station_name|count|
+--------------------+-----+
|San Francisco Cal...|49092|
|San Francisco Cal...|33742|
|Harry Bridges Pla...|32934|
|Embarcadero at Sa...|27713|
|Temporary Transba...|26089|
+--------------------+-----+
only showing top 5 rows
